# World Cup Match Outcome Predictor

This notebook contains the final modelling pipeline and a clearly separated retrospective validation against the completed 2026 FIFA World Cup. Historical model development remains unchanged through Commit 11. Commit 12 freezes that selected model and pre-tournament team state, then evaluates 104 tournament matches without retraining, retuning or reselecting the model.


In [ ]:
from pathlib import Path
import hashlib
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

# Resolve the project root whether Jupyter starts from the repository root
# or from inside the notebooks directory.
current_dir = Path.cwd()
if (current_dir / "data" / "results.csv").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "results.csv").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate data/results.csv. Start Jupyter from the project root "
        "or the notebooks directory."
    )

results_path = project_root / "data" / "results.csv"
outputs_dir = project_root / "outputs"
outputs_dir.mkdir(exist_ok=True)

required_columns = [
    "date", "home_team", "away_team", "home_score", "away_score",
    "tournament", "city", "country", "neutral"
]

df_raw = pd.read_csv(results_path)
missing_columns = sorted(set(required_columns) - set(df_raw.columns))
if missing_columns:
    raise ValueError(f"results.csv is missing required columns: {missing_columns}")

source_rows = len(df_raw)
df_matches = df_raw[required_columns].copy()
df_matches["date"] = pd.to_datetime(df_matches["date"], errors="coerce")
df_matches["home_score"] = pd.to_numeric(df_matches["home_score"], errors="coerce")
df_matches["away_score"] = pd.to_numeric(df_matches["away_score"], errors="coerce")

invalid_date_rows = int(df_matches["date"].isna().sum())
missing_score_rows = int(df_matches[["home_score", "away_score"]].isna().any(axis=1).sum())
exact_duplicate_rows = int(df_matches.duplicated(subset=required_columns, keep="first").sum())

# Keep only completed, valid, exact-unique matches.
df_matches = (
    df_matches
    .dropna(subset=["date", "home_score", "away_score"])
    .drop_duplicates(subset=required_columns, keep="first")
    .copy()
)

# Deterministic match identifier derived from the complete validated record.
def make_match_id(row):
    values = []
    for column in required_columns:
        value = row[column]
        if column == "date":
            value = value.strftime("%Y-%m-%d")
        elif isinstance(value, float) and value.is_integer():
            value = int(value)
        values.append(str(value).strip())
    payload = "|".join(values)
    return "match_" + hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

df_matches["match_id"] = df_matches.apply(make_match_id, axis=1)
if not df_matches["match_id"].is_unique:
    raise ValueError("Generated match_id values are not unique.")

# Report keys that are not guaranteed unique even though complete records differ.
duplicate_key_mask = df_matches.duplicated(
    subset=["date", "home_team", "away_team"], keep=False
)
duplicate_match_keys = df_matches.loc[
    duplicate_key_mask,
    ["match_id", "date", "home_team", "away_team", "home_score", "away_score", "tournament"]
].sort_values(["date", "home_team", "away_team"])
duplicate_match_keys.to_csv(outputs_dir / "duplicate_match_keys.csv", index=False)

validation_summary = pd.DataFrame([
    ("source_rows", source_rows),
    ("invalid_date_rows", invalid_date_rows),
    ("missing_score_rows_removed", missing_score_rows),
    ("exact_duplicate_rows_removed", exact_duplicate_rows),
    ("validated_completed_matches", len(df_matches)),
    ("unique_match_ids", df_matches["match_id"].nunique()),
    ("duplicate_date_home_away_rows", len(duplicate_match_keys)),
], columns=["metric", "value"])
validation_summary.to_csv(outputs_dir / "data_validation_summary.csv", index=False)

print("Project root:", project_root)
print("Dataset path:", results_path)
print("Validated completed matches:", len(df_matches))
print("Unique match IDs:", df_matches["match_id"].nunique())
display(validation_summary)
display(df_matches.head())


In [ ]:
# Filter to the modern modelling period after validation.
modern_era_cutoff = pd.Timestamp("2000-01-01")
df_clean = df_matches.loc[df_matches["date"] >= modern_era_cutoff].copy()

if not df_clean["match_id"].is_unique:
    raise ValueError("match_id must remain unique after filtering.")

print("Validated completed matches:", len(df_matches))
print("Total matches from 2000 onward:", len(df_clean))
print("Unique modern-era match IDs:", df_clean["match_id"].nunique())
print("\nTop 10 Tournament Types:")
print(df_clean["tournament"].value_counts().head(10))


In [ ]:
# Build one team-perspective row for each side of every validated match.
df_home = df_clean[[
    "match_id", "date", "home_team", "home_score", "away_score"
]].copy()
df_home["side"] = "home"
df_home = df_home.rename(columns={
    "home_team": "team",
    "home_score": "goals_for",
    "away_score": "goals_against",
})

df_away = df_clean[[
    "match_id", "date", "away_team", "away_score", "home_score"
]].copy()
df_away["side"] = "away"
df_away = df_away.rename(columns={
    "away_team": "team",
    "away_score": "goals_for",
    "home_score": "goals_against",
})

df_team_matches = (
    pd.concat([df_home, df_away], ignore_index=True)
    .sort_values(["team", "date", "match_id", "side"])
    .reset_index(drop=True)
)

expected_team_rows = 2 * len(df_clean)
if len(df_team_matches) != expected_team_rows:
    raise ValueError(
        f"Expected {expected_team_rows} team-perspective rows, found {len(df_team_matches)}."
    )

# Previous-five-match rolling form. shift(1) excludes the current result.
window_size = 5
df_team_matches["form_goals_for"] = (
    df_team_matches.groupby("team")["goals_for"]
    .transform(lambda s: s.shift(1).rolling(window_size, min_periods=1).mean())
)
df_team_matches["form_goals_against"] = (
    df_team_matches.groupby("team")["goals_against"]
    .transform(lambda s: s.shift(1).rolling(window_size, min_periods=1).mean())
)

# If one team has multiple source matches on the same date, no kickoff times
# are available. Give every same-day row the pre-date value from the first row
# so one same-day result cannot leak into another.
df_team_matches["form_goals_for"] = (
    df_team_matches.groupby(["team", "date"])["form_goals_for"].transform("first")
)
df_team_matches["form_goals_against"] = (
    df_team_matches.groupby(["team", "date"])["form_goals_against"].transform("first")
)

df_team_matches[["form_goals_for", "form_goals_against"]] = (
    df_team_matches[["form_goals_for", "form_goals_against"]].fillna(0.0)
)

same_day_rows = df_team_matches[
    df_team_matches.duplicated(["team", "date"], keep=False)
]
same_day_form_counts = same_day_rows.groupby(["team", "date"])[
    ["form_goals_for", "form_goals_against"]
].nunique()
if not same_day_form_counts.empty and (same_day_form_counts > 1).any().any():
    raise ValueError("Same-day matches received different pre-date form values.")

print("Team-perspective rows:", len(df_team_matches))
print("Expected team-perspective rows:", expected_team_rows)
print("Same-day team/date groups handled:", len(same_day_form_counts))
print("\nArgentina's leakage-safe form check:")
display(df_team_matches.loc[df_team_matches["team"].eq("Argentina")].tail(10))


In [ ]:
# Create one unambiguous form lookup for each side of every match.
home_form = (
    df_team_matches.loc[
        df_team_matches["side"].eq("home"),
        ["match_id", "form_goals_for", "form_goals_against"]
    ]
    .rename(columns={
        "form_goals_for": "home_form_goals_for",
        "form_goals_against": "home_form_goals_against",
    })
)

away_form = (
    df_team_matches.loc[
        df_team_matches["side"].eq("away"),
        ["match_id", "form_goals_for", "form_goals_against"]
    ]
    .rename(columns={
        "form_goals_for": "away_form_goals_for",
        "form_goals_against": "away_form_goals_against",
    })
)

if not home_form["match_id"].is_unique or not away_form["match_id"].is_unique:
    raise ValueError("Rolling-form lookups must contain one row per match_id.")

df_model = (
    df_clean
    .merge(home_form, on="match_id", how="left", validate="one_to_one")
    .merge(away_form, on="match_id", how="left", validate="one_to_one")
)

form_columns = [
    "home_form_goals_for", "home_form_goals_against",
    "away_form_goals_for", "away_form_goals_against",
]
if df_model[form_columns].isna().any().any():
    raise ValueError("Missing rolling-form values after match_id merge.")
if len(df_model) != len(df_clean) or not df_model["match_id"].is_unique:
    raise ValueError("Rolling-form merge changed the one-row-per-match structure.")

conditions = [
    df_model["home_score"] > df_model["away_score"],
    df_model["home_score"] == df_model["away_score"],
    df_model["home_score"] < df_model["away_score"],
]
df_model["target"] = np.select(conditions, [2, 1, 0], default=np.nan)
if df_model["target"].isna().any():
    raise ValueError("Target encoding produced missing values.")

rolling_form_summary = pd.DataFrame([
    ("modern_matches", len(df_clean)),
    ("team_perspective_rows", len(df_team_matches)),
    ("expected_team_perspective_rows", 2 * len(df_clean)),
    ("same_day_team_date_groups", len(same_day_form_counts)),
    ("home_form_lookup_rows", len(home_form)),
    ("away_form_lookup_rows", len(away_form)),
    ("final_model_rows", len(df_model)),
    ("unique_model_match_ids", df_model["match_id"].nunique()),
    ("duplicate_model_match_ids", int(df_model["match_id"].duplicated().sum())),
    ("rows_added_or_lost", len(df_model) - len(df_clean)),
    ("missing_form_values", int(df_model[form_columns].isna().sum().sum())),
], columns=["metric", "value"])
rolling_form_summary.to_csv(outputs_dir / "rolling_form_validation.csv", index=False)

print("Final feature matrix rows:", len(df_model))
print("Unique match IDs:", df_model["match_id"].nunique())
print("Rows added or lost:", len(df_model) - len(df_clean))
display(rolling_form_summary)


In [ ]:
# Commit 5: corrected tournament importance and match-context features.
def normalize_tournament_name(tournament_name):
    # casefold handles case consistently; NFKD normalization removes accents,
    # so "Copa América" and "Copa America" are treated identically.
    text = str(tournament_name).casefold()
    return "".join(
        character for character in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(character)
    )


def get_tournament_weight(tournament_name):
    name = normalize_tournament_name(tournament_name)

    if "fifa world cup" in name and "qualification" not in name:
        return 1.0

    # Preserve the original prototype's intended high-weight competitions,
    # but match them explicitly so unrelated strings containing "Euro" are
    # not accidentally treated as UEFA European Championship finals.
    major_finals = (
        "confederations cup",
        "copa america",
        "uefa euro",
    )
    if any(term in name for term in major_finals) and "qualification" not in name:
        return 0.8

    if "qualification" in name:
        return 0.6
    if "nations league" in name:
        return 0.5
    if "friendly" in name:
        return 0.25
    return 0.4


def get_original_tournament_weight(tournament_name):
    # Reproduce the pre-Commit-5 rules for validation only.
    if "FIFA World Cup" in tournament_name and "qualification" not in tournament_name:
        return 1.0
    elif (
        "Confederations Cup" in tournament_name
        or "Copa America" in tournament_name
        or "Euro" in tournament_name and "qualification" not in tournament_name
    ):
        return 0.8
    elif "qualification" in tournament_name:
        return 0.6
    elif "Nations League" in tournament_name:
        return 0.5
    elif "Friendly" in tournament_name:
        return 0.25
    return 0.4


# Add context directly to the one-row-per-match modelling table. Do not rebuild
# the rolling-form merge.
df_model["match_weight"] = df_model["tournament"].apply(get_tournament_weight)
df_model["is_neutral"] = df_model["neutral"].astype(int)
df_clean["match_weight"] = df_clean["tournament"].apply(get_tournament_weight)
df_clean["is_neutral"] = df_clean["neutral"].astype(int)

if len(df_model) != len(df_clean) or not df_model["match_id"].is_unique:
    raise ValueError("Match-context features changed the one-row-per-match structure.")
if not set(df_model["is_neutral"].unique()).issubset({0, 1}):
    raise ValueError("Neutral-venue feature must contain only 0 and 1.")

# Validate the bug fix against the actual dataset.
weight_audit = df_clean[["tournament"]].copy()
weight_audit["original_weight"] = weight_audit["tournament"].apply(get_original_tournament_weight)
weight_audit["corrected_weight"] = weight_audit["tournament"].apply(get_tournament_weight)

weight_validation = (
    weight_audit.groupby(["tournament", "original_weight", "corrected_weight"])
    .size()
    .reset_index(name="modern_match_count")
    .sort_values(["corrected_weight", "modern_match_count", "tournament"], ascending=[False, False, True])
)
weight_validation["weight_changed"] = (
    weight_validation["original_weight"] != weight_validation["corrected_weight"]
)
weight_validation.to_csv(outputs_dir / "tournament_weight_validation.csv", index=False)

changed_weight_rows = int((weight_audit["original_weight"] != weight_audit["corrected_weight"]).sum())
copa_america_rows = int((
    df_clean["tournament"].map(normalize_tournament_name) == "copa america"
).sum())
copa_america_correct = int((
    (df_clean["tournament"].map(normalize_tournament_name) == "copa america")
    & (df_clean["match_weight"] == 0.8)
).sum())

context_summary = pd.DataFrame([
    ("modern_matches", len(df_clean)),
    ("rows_changed_vs_original_weight_rules", changed_weight_rows),
    ("copa_america_matches", copa_america_rows),
    ("copa_america_matches_weighted_0_8", copa_america_correct),
    ("world_cup_final_weight_1_0", int((df_clean["match_weight"] == 1.0).sum())),
    ("major_final_weight_0_8", int((df_clean["match_weight"] == 0.8).sum())),
    ("qualification_weight_0_6", int((df_clean["match_weight"] == 0.6).sum())),
    ("nations_league_weight_0_5", int((df_clean["match_weight"] == 0.5).sum())),
    ("other_weight_0_4", int((df_clean["match_weight"] == 0.4).sum())),
    ("friendly_weight_0_25", int((df_clean["match_weight"] == 0.25).sum())),
    ("neutral_matches", int(df_clean["is_neutral"].sum())),
], columns=["metric", "value"])
context_summary.to_csv(outputs_dir / "match_context_validation.csv", index=False)

if copa_america_rows != copa_america_correct:
    raise ValueError("Not all Copa América finals received weight 0.8.")

features = [
    "home_form_goals_for", "home_form_goals_against",
    "away_form_goals_for", "away_form_goals_against",
    "match_weight", "is_neutral",
]
X = df_model[features]
y = df_model["target"]

print("Corrected tournament weights added without rebuilding the feature merge.")
print("Copa América matches correctly weighted 0.8:", copa_america_correct)
print("Rows whose weight changed vs original rules:", changed_weight_rows)
print("Feature matrix shape:", X.shape)
display(context_summary)
display(weight_validation.loc[weight_validation["weight_changed"]])


In [ ]:
# Chronological forecast evaluation.
evaluation_cutoff = pd.Timestamp('2023-01-01')

train_mask = df_model['date'] < evaluation_cutoff
test_mask = df_model['date'] >= evaluation_cutoff

y_train = df_model.loc[train_mask, 'target'].copy()
y_test = df_model.loc[test_mask, 'target'].copy()

training_rows = int(train_mask.sum())
test_rows = int(test_mask.sum())

if training_rows == 0 or test_rows == 0:
    raise ValueError('Chronological split produced an empty training or test set.')
if training_rows + test_rows != len(df_model):
    raise ValueError('Chronological split does not cover every modelling row exactly once.')

train_date_min = df_model.loc[train_mask, 'date'].min()
train_date_max = df_model.loc[train_mask, 'date'].max()
test_date_min = df_model.loc[test_mask, 'date'].min()
test_date_max = df_model.loc[test_mask, 'date'].max()

if train_date_max >= test_date_min:
    raise ValueError('Temporal leakage detected: training dates overlap the test period.')

split_summary = pd.DataFrame([
    ('evaluation_cutoff', evaluation_cutoff.strftime('%Y-%m-%d')),
    ('total_model_rows', len(df_model)),
    ('training_rows', training_rows),
    ('test_rows', test_rows),
    ('training_start', train_date_min.strftime('%Y-%m-%d')),
    ('training_end', train_date_max.strftime('%Y-%m-%d')),
    ('test_start', test_date_min.strftime('%Y-%m-%d')),
    ('test_end', test_date_max.strftime('%Y-%m-%d')),
    ('date_overlap_detected', int(train_date_max >= test_date_min)),
], columns=['metric', 'value'])
split_summary.to_csv(outputs_dir / 'chronological_split_validation.csv', index=False)

class_distribution = pd.DataFrame({
    'training_count': y_train.value_counts().reindex([0, 1, 2], fill_value=0),
    'test_count': y_test.value_counts().reindex([0, 1, 2], fill_value=0),
}).astype(int)
class_distribution.index.name = 'target'
class_distribution.to_csv(outputs_dir / 'chronological_split_class_distribution.csv')

print('Chronological evaluation split')
print('Training:', train_date_min.date(), 'to', train_date_max.date(), '-', training_rows, 'matches')
print('Testing: ', test_date_min.date(), 'to', test_date_max.date(), '-', test_rows, 'matches')
display(split_summary)
display(class_distribution)


In [ ]:
# Commit 6: deterministic pre-match Elo features keyed by match_id.
def get_expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def elo_rating_change(rating, expected, actual, k):
    return k * (actual - expected)


# Baseline Elo for a team at its first modern-era appearance.
elo_baseline = 1500.0
current_elo = {}
elo_feature_rows = []

# Process one calendar date at a time. If a team appears more than once on
# the same date, every match on that date must see the same pre-date Elo.
# Daily rating changes are therefore accumulated and applied only after all
# matches on that date have been evaluated.
elo_source = df_clean.sort_values(["date", "match_id"]).copy()

print("Calculating leakage-safe historical Elo ratings...")
for match_date, day_matches in elo_source.groupby("date", sort=True):
    pre_date_ratings = dict(current_elo)
    daily_changes = {}

    for row in day_matches.itertuples(index=False):
        home_rating = float(pre_date_ratings.get(row.home_team, elo_baseline))
        away_rating = float(pre_date_ratings.get(row.away_team, elo_baseline))

        elo_feature_rows.append({
            "match_id": row.match_id,
            "home_elo": home_rating,
            "away_elo": away_rating,
        })

        if row.home_score > row.away_score:
            actual_home, actual_away = 1.0, 0.0
        elif row.home_score < row.away_score:
            actual_home, actual_away = 0.0, 1.0
        else:
            actual_home, actual_away = 0.5, 0.5

        expected_home = get_expected_score(home_rating, away_rating)
        expected_away = get_expected_score(away_rating, home_rating)
        k_adjusted = 30.0 * float(row.match_weight)

        home_change = elo_rating_change(
            home_rating, expected_home, actual_home, k_adjusted
        )
        away_change = elo_rating_change(
            away_rating, expected_away, actual_away, k_adjusted
        )

        daily_changes[row.home_team] = daily_changes.get(row.home_team, 0.0) + home_change
        daily_changes[row.away_team] = daily_changes.get(row.away_team, 0.0) + away_change

    # Apply the whole date batch after every match has received pre-date Elo.
    teams_today = set(day_matches["home_team"]) | set(day_matches["away_team"])
    for team in teams_today:
        starting_rating = float(pre_date_ratings.get(team, elo_baseline))
        current_elo[team] = starting_rating + daily_changes.get(team, 0.0)

elo_features = pd.DataFrame(elo_feature_rows)

if len(elo_features) != len(df_clean):
    raise ValueError(
        f"Expected {len(df_clean)} Elo feature rows, found {len(elo_features)}."
    )
if not elo_features["match_id"].is_unique:
    raise ValueError("Elo feature table must contain one row per match_id.")
if set(elo_features["match_id"]) != set(df_clean["match_id"]):
    raise ValueError("Elo feature match_id values do not match the validated data.")

# Verify repeated team/date appearances received identical pre-date ratings.
elo_audit = df_clean[["match_id", "date", "home_team", "away_team"]].merge(
    elo_features, on="match_id", how="left", validate="one_to_one"
)
home_elo_audit = elo_audit[["date", "home_team", "home_elo"]].rename(
    columns={"home_team": "team", "home_elo": "elo"}
)
away_elo_audit = elo_audit[["date", "away_team", "away_elo"]].rename(
    columns={"away_team": "team", "away_elo": "elo"}
)
team_date_elo_audit = pd.concat([home_elo_audit, away_elo_audit], ignore_index=True)
repeated_team_dates = team_date_elo_audit[
    team_date_elo_audit.duplicated(["team", "date"], keep=False)
]
repeated_rating_counts = repeated_team_dates.groupby(["team", "date"])["elo"].nunique()
if not repeated_rating_counts.empty and (repeated_rating_counts > 1).any():
    raise ValueError("Same-day matches received different pre-date Elo ratings.")

final_elo_ratings = (
    pd.DataFrame(current_elo.items(), columns=["team", "final_elo"])
    .sort_values("final_elo", ascending=False)
    .reset_index(drop=True)
)
final_elo_ratings.to_csv(outputs_dir / "final_elo_ratings.csv", index=False)

print("Elo feature rows:", len(elo_features))
print("Unique Elo match IDs:", elo_features["match_id"].nunique())
print("Repeated team/date groups handled:", len(repeated_rating_counts))
print("\nTop 5 Teams by Final Elo Rating:")
display(final_elo_ratings.head(5))


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt

# Join Elo features by the permanent match identifier.
model_rows_before_elo = len(df_model)
df_model = df_model.merge(
    elo_features,
    on='match_id',
    how='left',
    validate='one_to_one',
)

elo_columns = ['home_elo', 'away_elo']
missing_elo_values = int(df_model[elo_columns].isna().sum().sum())

if missing_elo_values:
    raise ValueError(f'Elo merge produced {missing_elo_values} missing values.')
if len(df_model) != model_rows_before_elo:
    raise ValueError(
        f'Elo merge changed model rows from {model_rows_before_elo} to {len(df_model)}.'
    )
if not df_model['match_id'].is_unique:
    raise ValueError('df_model contains duplicate match_id values after Elo merge.')

elo_validation = pd.DataFrame([
    ('modern_matches', len(df_clean)),
    ('elo_feature_rows', len(elo_features)),
    ('unique_elo_match_ids', elo_features['match_id'].nunique()),
    ('model_rows_before_elo_merge', model_rows_before_elo),
    ('model_rows_after_elo_merge', len(df_model)),
    ('duplicate_model_match_ids', int(df_model['match_id'].duplicated().sum())),
    ('missing_elo_values', missing_elo_values),
    ('repeated_team_date_groups', len(repeated_rating_counts)),
    ('teams_with_final_elo', len(final_elo_ratings)),
], columns=['metric', 'value'])
elo_validation.to_csv(outputs_dir / 'elo_validation.csv', index=False)

features_upgraded = [
    'home_elo',
    'away_elo',
    'home_form_goals_for',
    'home_form_goals_against',
    'away_form_goals_for',
    'away_form_goals_against',
    'match_weight',
    'is_neutral',
]

train_mask_up = df_model['date'] < evaluation_cutoff
test_mask_up = df_model['date'] >= evaluation_cutoff

X_train_up = df_model.loc[train_mask_up, features_upgraded].copy()
X_test_up = df_model.loc[test_mask_up, features_upgraded].copy()
y_train_up = df_model.loc[train_mask_up, 'target'].copy()
y_test_up = df_model.loc[test_mask_up, 'target'].copy()

if len(X_train_up) != training_rows or len(X_test_up) != test_rows:
    raise ValueError('Elo model does not use the expected chronological split.')
if df_model.loc[train_mask_up, 'date'].max() >= df_model.loc[test_mask_up, 'date'].min():
    raise ValueError('Temporal leakage detected after the Elo merge.')

X_upgraded = df_model[features_upgraded]
y_upgraded = df_model['target']

class_labels = np.array([0, 1, 2])
class_names = {
    0: 'Away Win',
    1: 'Draw',
    2: 'Home Win',
}


class BalancedHistGradientBoostingClassifier(ClassifierMixin, BaseEstimator):
    """HistGradientBoostingClassifier with balanced weights computed inside fit.

    Keeping the weighting inside the estimator lets CalibratedClassifierCV train
    the classifier with class balancing while fitting the calibration layer on
    the natural, unweighted calibration distribution.
    """

    def __init__(
        self,
        learning_rate=0.08,
        max_iter=200,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    ):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.max_leaf_nodes = max_leaf_nodes
        self.l2_regularization = l2_regularization
        self.random_state = random_state

    def fit(self, X, y):
        self.model_ = HistGradientBoostingClassifier(
            learning_rate=self.learning_rate,
            max_iter=self.max_iter,
            max_leaf_nodes=self.max_leaf_nodes,
            l2_regularization=self.l2_regularization,
            random_state=self.random_state,
        )

        sample_weights = compute_sample_weight(
            class_weight='balanced',
            y=y,
        )

        self.model_.fit(
            X,
            y,
            sample_weight=sample_weights,
        )
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


# -------------------------------------------------------------------------
# Classification diagnostics
# -------------------------------------------------------------------------

scaler_upgraded = StandardScaler()
X_train_scaled_up = scaler_upgraded.fit_transform(X_train_up)
X_test_scaled_up = scaler_upgraded.transform(X_test_up)

dummy_model = DummyClassifier(
    strategy='prior',
    random_state=42,
)
linear_svm_model = LinearSVC(
    class_weight='balanced',
    random_state=42,
    max_iter=5000,
)
balanced_hgb_model = BalancedHistGradientBoostingClassifier()

print('Training class-prior baseline...')
dummy_model.fit(X_train_up, y_train_up)

print('Training class-balanced Linear SVM...')
linear_svm_model.fit(X_train_scaled_up, y_train_up)

print('Training class-balanced HistGradientBoosting...')
balanced_hgb_model.fit(X_train_up, y_train_up)

classification_models = {
    'Class-prior baseline': (dummy_model, X_test_up),
    'Linear SVM': (linear_svm_model, X_test_scaled_up),
    'Balanced HistGradientBoosting': (balanced_hgb_model, X_test_up),
}

comparison_rows = []
model_predictions = {}
confusion_rows = []
recall_rows = []

for model_name, (model, model_X_test) in classification_models.items():
    predictions = model.predict(model_X_test)
    model_predictions[model_name] = predictions

    comparison_rows.append({
        'model': model_name,
        'accuracy': accuracy_score(y_test_up, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_test_up, predictions),
        'macro_f1': f1_score(y_test_up, predictions, average='macro'),
    })

    recalls = recall_score(
        y_test_up,
        predictions,
        labels=class_labels,
        average=None,
        zero_division=0,
    )

    for label, recall_value in zip(class_labels, recalls):
        recall_rows.append({
            'model': model_name,
            'class_label': int(label),
            'class_name': class_names[int(label)],
            'recall': float(recall_value),
        })

    matrix = confusion_matrix(
        y_test_up,
        predictions,
        labels=class_labels,
    )

    for actual_index, actual_label in enumerate(class_labels):
        for predicted_index, predicted_label in enumerate(class_labels):
            confusion_rows.append({
                'model': model_name,
                'actual_label': int(actual_label),
                'actual_name': class_names[int(actual_label)],
                'predicted_label': int(predicted_label),
                'predicted_name': class_names[int(predicted_label)],
                'count': int(matrix[actual_index, predicted_index]),
            })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        ['macro_f1', 'balanced_accuracy'],
        ascending=False,
    )
    .reset_index(drop=True)
)
model_comparison.to_csv(
    outputs_dir / 'model_comparison_classification.csv',
    index=False,
)

per_class_recall = pd.DataFrame(recall_rows)
per_class_recall.to_csv(
    outputs_dir / 'model_per_class_recall.csv',
    index=False,
)

model_confusion_matrices = pd.DataFrame(confusion_rows)
model_confusion_matrices.to_csv(
    outputs_dir / 'model_confusion_matrices.csv',
    index=False,
)

print('\nClassification comparison')
display(model_comparison)

print('\nDraw recall from hard class predictions')
display(
    per_class_recall.loc[
        per_class_recall['class_label'].eq(1),
        ['model', 'recall'],
    ]
    .sort_values('recall', ascending=False)
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Temporal calibration folds
# -------------------------------------------------------------------------

X_train_probability = X_train_up.reset_index(drop=True)
y_train_probability = y_train_up.reset_index(drop=True)
X_test_probability = X_test_up.reset_index(drop=True)
y_test_probability = y_test_up.reset_index(drop=True)

training_dates_probability = (
    df_model.loc[train_mask_up, 'date']
    .reset_index(drop=True)
)

if not training_dates_probability.is_monotonic_increasing:
    raise ValueError(
        'Training rows are not ordered chronologically; temporal calibration would be invalid.'
    )

unique_training_dates = np.array(
    sorted(training_dates_probability.unique())
)
date_blocks = np.array_split(
    unique_training_dates,
    6,
)

temporal_calibration_folds = []
calibration_fold_rows = []

for fold_number in range(1, 6):
    estimator_dates = np.concatenate(
        date_blocks[:fold_number]
    )
    calibration_dates = date_blocks[fold_number]

    estimator_idx = np.flatnonzero(
        training_dates_probability
        .isin(estimator_dates)
        .to_numpy()
    )
    calibration_idx = np.flatnonzero(
        training_dates_probability
        .isin(calibration_dates)
        .to_numpy()
    )

    estimator_end = pd.Timestamp(estimator_dates[-1])
    calibration_start = pd.Timestamp(calibration_dates[0])

    if estimator_end >= calibration_start:
        raise ValueError(
            f'Calibration fold {fold_number} is not strictly chronological.'
        )

    estimator_classes = set(
        y_train_probability
        .iloc[estimator_idx]
        .astype(int)
        .unique()
    )
    calibration_classes = set(
        y_train_probability
        .iloc[calibration_idx]
        .astype(int)
        .unique()
    )
    expected_classes = set(class_labels.tolist())

    if estimator_classes != expected_classes:
        raise ValueError(
            f'Estimator block {fold_number} is missing an outcome class.'
        )
    if calibration_classes != expected_classes:
        raise ValueError(
            f'Calibration block {fold_number} is missing an outcome class.'
        )

    temporal_calibration_folds.append(
        (estimator_idx, calibration_idx)
    )
    calibration_fold_rows.append({
        'fold': fold_number,
        'estimator_rows': len(estimator_idx),
        'calibration_rows': len(calibration_idx),
        'estimator_start': pd.Timestamp(estimator_dates[0]).strftime('%Y-%m-%d'),
        'estimator_end': estimator_end.strftime('%Y-%m-%d'),
        'calibration_start': calibration_start.strftime('%Y-%m-%d'),
        'calibration_end': pd.Timestamp(calibration_dates[-1]).strftime('%Y-%m-%d'),
        'date_overlap_detected': 0,
    })

calibration_fold_validation = pd.DataFrame(
    calibration_fold_rows
)
calibration_fold_validation.to_csv(
    outputs_dir / 'calibration_fold_validation.csv',
    index=False,
)


# -------------------------------------------------------------------------
# Final calibrated probability candidates
# -------------------------------------------------------------------------

linear_svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC(
        class_weight='balanced',
        random_state=42,
        max_iter=5000,
    )),
])

try:
    calibrated_linear_svm_model = CalibratedClassifierCV(
        estimator=linear_svm_pipeline,
        method='sigmoid',
        cv=temporal_calibration_folds,
        ensemble=True,
    )
    calibrated_balanced_hgb_model = CalibratedClassifierCV(
        estimator=BalancedHistGradientBoostingClassifier(),
        method='sigmoid',
        cv=temporal_calibration_folds,
        ensemble=True,
    )
except TypeError:
    calibrated_linear_svm_model = CalibratedClassifierCV(
        base_estimator=linear_svm_pipeline,
        method='sigmoid',
        cv=temporal_calibration_folds,
        ensemble=True,
    )
    calibrated_balanced_hgb_model = CalibratedClassifierCV(
        base_estimator=BalancedHistGradientBoostingClassifier(),
        method='sigmoid',
        cv=temporal_calibration_folds,
        ensemble=True,
    )

print('Training temporally calibrated Linear SVM...')
calibrated_linear_svm_model.fit(
    X_train_probability,
    y_train_probability,
)

print('Training temporally calibrated balanced HistGradientBoosting...')
calibrated_balanced_hgb_model.fit(
    X_train_probability,
    y_train_probability,
)


def align_probability_columns(model, probabilities, labels):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )
    aligned = np.zeros(
        (len(probabilities), len(labels)),
        dtype=float,
    )

    for source_index, class_label in enumerate(model.classes_):
        destination = np.where(
            labels == int(class_label)
        )[0]

        if len(destination) != 1:
            raise ValueError(
                f'Unexpected model class label: {class_label}'
            )

        aligned[:, destination[0]] = probabilities[:, source_index]

    row_sums = aligned.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        raise ValueError(
            'Predicted probabilities do not sum to one.'
        )

    return aligned


def multiclass_brier_score(
    y_true,
    probabilities,
    labels,
):
    y_array = np.asarray(
        y_true,
        dtype=int,
    )
    one_hot = np.column_stack([
        (y_array == label).astype(float)
        for label in labels
    ])

    return float(
        np.mean(
            np.sum(
                (probabilities - one_hot) ** 2,
                axis=1,
            )
        )
    )


probability_models = {
    'Class-prior baseline': dummy_model,
    'Calibrated Linear SVM': calibrated_linear_svm_model,
    'Calibrated Balanced HistGradientBoosting': calibrated_balanced_hgb_model,
}

probability_comparison_rows = []
probability_predictions = {}
probability_outputs = {}

for model_name, model in probability_models.items():
    raw_probabilities = model.predict_proba(
        X_test_probability
    )
    probabilities = align_probability_columns(
        model,
        raw_probabilities,
        class_labels,
    )
    predictions = class_labels[
        np.argmax(probabilities, axis=1)
    ]

    probability_predictions[model_name] = predictions
    probability_outputs[model_name] = probabilities

    recalls = recall_score(
        y_test_probability,
        predictions,
        labels=class_labels,
        average=None,
        zero_division=0,
    )

    probability_comparison_rows.append({
        'model': model_name,
        'accuracy': accuracy_score(
            y_test_probability,
            predictions,
        ),
        'balanced_accuracy': balanced_accuracy_score(
            y_test_probability,
            predictions,
        ),
        'macro_f1': f1_score(
            y_test_probability,
            predictions,
            average='macro',
        ),
        'away_win_recall': float(recalls[0]),
        'draw_recall': float(recalls[1]),
        'home_win_recall': float(recalls[2]),
        'log_loss': log_loss(
            y_test_probability,
            probabilities,
            labels=class_labels,
        ),
        'multiclass_brier': multiclass_brier_score(
            y_test_probability,
            probabilities,
            class_labels,
        ),
    })

probability_model_comparison = (
    pd.DataFrame(probability_comparison_rows)
    .sort_values(
        ['log_loss', 'multiclass_brier'],
        ascending=True,
    )
    .reset_index(drop=True)
)
probability_model_comparison.insert(
    0,
    'probability_rank',
    np.arange(
        1,
        len(probability_model_comparison) + 1,
    ),
)
probability_model_comparison.to_csv(
    outputs_dir / 'probability_model_comparison.csv',
    index=False,
)

selected_probability_model_name = (
    probability_model_comparison.loc[0, 'model']
)
selected_probability_model = probability_models[
    selected_probability_model_name
]
selected_test_probabilities = probability_outputs[
    selected_probability_model_name
]
selected_test_predictions = probability_predictions[
    selected_probability_model_name
]

selected_model_summary = (
    probability_model_comparison
    .iloc[[0]]
    .copy()
)
selected_model_summary.to_csv(
    outputs_dir / 'selected_probability_model.csv',
    index=False,
)

print('\nFinal calibrated probability comparison')
display(probability_model_comparison)
print(
    'Selected probability model:',
    selected_probability_model_name,
)


# -------------------------------------------------------------------------
# Calibration and draw-probability diagnostics
# -------------------------------------------------------------------------

calibration_rows = []

for class_index, class_label in enumerate(class_labels):
    observed_frequency, mean_predicted_probability = calibration_curve(
        (
            np.asarray(
                y_test_probability,
                dtype=int,
            )
            == class_label
        ).astype(int),
        selected_test_probabilities[:, class_index],
        n_bins=10,
        strategy='quantile',
    )

    for bin_number, (predicted, observed) in enumerate(
        zip(
            mean_predicted_probability,
            observed_frequency,
        ),
        start=1,
    ):
        calibration_rows.append({
            'model': selected_probability_model_name,
            'class_label': int(class_label),
            'class_name': class_names[int(class_label)],
            'bin': bin_number,
            'mean_predicted_probability': float(predicted),
            'observed_frequency': float(observed),
        })

selected_model_calibration = pd.DataFrame(
    calibration_rows
)
selected_model_calibration.to_csv(
    outputs_dir / 'selected_model_calibration.csv',
    index=False,
)

draw_curve = selected_model_calibration.loc[
    selected_model_calibration['class_label'].eq(1)
].copy()

selected_draw_recall = recall_score(
    y_test_probability,
    selected_test_predictions,
    labels=class_labels,
    average=None,
    zero_division=0,
)[1]

draw_probability_diagnostic = pd.DataFrame([
    {
        'model': selected_probability_model_name,
        'test_matches': len(y_test_probability),
        'actual_draws': int(
            (np.asarray(y_test_probability) == 1).sum()
        ),
        'actual_draw_rate': float(
            (np.asarray(y_test_probability) == 1).mean()
        ),
        'mean_predicted_draw_probability': float(
            selected_test_probabilities[:, 1].mean()
        ),
        'argmax_draw_prediction_rate': float(
            (selected_test_predictions == 1).mean()
        ),
        'argmax_draw_recall': float(selected_draw_recall),
        'draw_calibration_bin_mae': float(
            np.mean(
                np.abs(
                    draw_curve['mean_predicted_probability']
                    - draw_curve['observed_frequency']
                )
            )
        ),
    }
])
draw_probability_diagnostic.to_csv(
    outputs_dir / 'draw_probability_diagnostic.csv',
    index=False,
)

# Compare the raw balanced HGB probabilities from the classification model with
# the fair, temporally calibrated HGB probabilities.
raw_hgb_probabilities = align_probability_columns(
    balanced_hgb_model,
    balanced_hgb_model.predict_proba(
        X_test_probability
    ),
    class_labels,
)
raw_hgb_predictions = class_labels[
    np.argmax(raw_hgb_probabilities, axis=1)
]
raw_hgb_recalls = recall_score(
    y_test_probability,
    raw_hgb_predictions,
    labels=class_labels,
    average=None,
    zero_division=0,
)

calibrated_hgb_row = (
    probability_model_comparison.loc[
        probability_model_comparison['model'].eq(
            'Calibrated Balanced HistGradientBoosting'
        )
    ]
    .iloc[0]
)

hgb_calibration_sensitivity = pd.DataFrame([
    {
        'variant': 'Balanced HGB raw probabilities',
        'accuracy': accuracy_score(
            y_test_probability,
            raw_hgb_predictions,
        ),
        'macro_f1': f1_score(
            y_test_probability,
            raw_hgb_predictions,
            average='macro',
        ),
        'draw_recall': float(raw_hgb_recalls[1]),
        'log_loss': log_loss(
            y_test_probability,
            raw_hgb_probabilities,
            labels=class_labels,
        ),
        'multiclass_brier': multiclass_brier_score(
            y_test_probability,
            raw_hgb_probabilities,
            class_labels,
        ),
    },
    {
        'variant': 'Balanced HGB temporally calibrated',
        'accuracy': float(calibrated_hgb_row['accuracy']),
        'macro_f1': float(calibrated_hgb_row['macro_f1']),
        'draw_recall': float(calibrated_hgb_row['draw_recall']),
        'log_loss': float(calibrated_hgb_row['log_loss']),
        'multiclass_brier': float(
            calibrated_hgb_row['multiclass_brier']
        ),
    },
])
hgb_calibration_sensitivity.to_csv(
    outputs_dir / 'hgb_calibration_sensitivity.csv',
    index=False,
)

selected_confusion = confusion_matrix(
    y_test_probability,
    selected_test_predictions,
    labels=class_labels,
)
selected_confusion_rows = []

for actual_index, actual_label in enumerate(class_labels):
    for predicted_index, predicted_label in enumerate(class_labels):
        selected_confusion_rows.append({
            'model': selected_probability_model_name,
            'actual_label': int(actual_label),
            'actual_name': class_names[int(actual_label)],
            'predicted_label': int(predicted_label),
            'predicted_name': class_names[int(predicted_label)],
            'count': int(
                selected_confusion[
                    actual_index,
                    predicted_index,
                ]
            ),
        })

pd.DataFrame(
    selected_confusion_rows
).to_csv(
    outputs_dir / 'selected_probability_confusion_matrix.csv',
    index=False,
)

figures_dir = outputs_dir / 'figures'
figures_dir.mkdir(exist_ok=True)

plt.figure(figsize=(7, 6))
plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    label='Perfect calibration',
)

for class_label in class_labels:
    class_curve = selected_model_calibration.loc[
        selected_model_calibration['class_label'].eq(
            int(class_label)
        )
    ]

    plt.plot(
        class_curve['mean_predicted_probability'],
        class_curve['observed_frequency'],
        marker='o',
        label=class_names[int(class_label)],
    )

plt.xlabel('Mean predicted probability')
plt.ylabel('Observed frequency')
plt.title(
    f'Calibration: {selected_probability_model_name}'
)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig(
    figures_dir / 'selected_model_calibration.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

plt.figure(figsize=(8, 5))
plot_frame = (
    probability_model_comparison
    .sort_values('log_loss', ascending=False)
)
plt.barh(
    plot_frame['model'],
    plot_frame['log_loss'],
)
plt.xlabel('Multiclass log loss (lower is better)')
plt.ylabel('Model')
plt.title('Final probability model comparison')
plt.tight_layout()
plt.savefig(
    figures_dir / 'final_log_loss_comparison.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

plt.figure(figsize=(8, 5))
draw_plot = (
    per_class_recall.loc[
        per_class_recall['class_label'].eq(1)
    ]
    .sort_values('recall')
)
plt.barh(
    draw_plot['model'],
    draw_plot['recall'] * 100,
)
plt.xlabel('Draw recall from argmax class prediction (%)')
plt.ylabel('Model')
plt.title('Draw detection in classification diagnostics')
plt.tight_layout()
plt.savefig(
    figures_dir / 'classification_draw_recall.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()


In [ ]:
from collections import Counter, defaultdict
from functools import lru_cache
from itertools import combinations
import difflib
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# Commit 10: build current team states from the actual historical data.
# -------------------------------------------------------------------------

# Current Elo is the post-match rating after the latest completed match in df_clean.
elo_state = pd.DataFrame(
    current_elo.items(),
    columns=['team', 'current_elo'],
)

# For a future match, form should include the most recent completed results rather
# than the pre-match values used for historical training rows.
recent_completed_matches = (
    df_team_matches
    .sort_values(['team', 'date', 'match_id', 'side'])
    .groupby('team', group_keys=False)
    .tail(window_size)
)

current_form = (
    recent_completed_matches
    .groupby('team', as_index=False)
    .agg(
        current_form_goals_for=('goals_for', 'mean'),
        current_form_goals_against=('goals_against', 'mean'),
        form_matches_used=('match_id', 'count'),
        latest_match_date=('date', 'max'),
    )
)

team_state_snapshot = (
    elo_state
    .merge(current_form, on='team', how='inner', validate='one_to_one')
    .sort_values('current_elo', ascending=False)
    .reset_index(drop=True)
)

if team_state_snapshot['team'].duplicated().any():
    raise ValueError('Team-state snapshot contains duplicate team names.')
if team_state_snapshot[[
    'current_elo',
    'current_form_goals_for',
    'current_form_goals_against',
]].isna().any().any():
    raise ValueError('Team-state snapshot contains missing model inputs.')

team_state_snapshot.to_csv(
    outputs_dir / 'team_state_snapshot.csv',
    index=False,
)

team_state_lookup = team_state_snapshot.set_index('team')
available_teams = set(team_state_lookup.index)


def _validate_team_name(team):
    if team in available_teams:
        return

    suggestions = difflib.get_close_matches(
        str(team),
        sorted(available_teams),
        n=5,
        cutoff=0.5,
    )
    suggestion_text = (
        f" Similar names: {', '.join(suggestions)}."
        if suggestions else ''
    )
    raise KeyError(
        f"Team '{team}' is not available in the current state snapshot."
        + suggestion_text
    )


def build_match_features(
    home_team,
    away_team,
    tournament_weight=1.0,
    neutral=True,
):
    _validate_team_name(home_team)
    _validate_team_name(away_team)

    if home_team == away_team:
        raise ValueError('A team cannot play itself.')

    home = team_state_lookup.loc[home_team]
    away = team_state_lookup.loc[away_team]

    feature_row = pd.DataFrame([{
        'home_elo': float(home['current_elo']),
        'away_elo': float(away['current_elo']),
        'home_form_goals_for': float(home['current_form_goals_for']),
        'home_form_goals_against': float(home['current_form_goals_against']),
        'away_form_goals_for': float(away['current_form_goals_for']),
        'away_form_goals_against': float(away['current_form_goals_against']),
        'match_weight': float(tournament_weight),
        'is_neutral': int(bool(neutral)),
    }], columns=features_upgraded)

    return feature_row


def _ordered_probability_vector(
    home_team,
    away_team,
    tournament_weight,
    neutral,
):
    features_for_match = build_match_features(
        home_team,
        away_team,
        tournament_weight=tournament_weight,
        neutral=neutral,
    )
    raw_probabilities = selected_probability_model.predict_proba(
        features_for_match
    )
    return align_probability_columns(
        selected_probability_model,
        raw_probabilities,
        class_labels,
    )[0]


@lru_cache(maxsize=None)
def _cached_team_probabilities(
    home_team,
    away_team,
    tournament_weight=1.0,
    neutral=True,
):
    direct = _ordered_probability_vector(
        home_team,
        away_team,
        tournament_weight,
        neutral,
    )

    # direct class order:
    # 0 = away win, 1 = draw, 2 = home win
    away_win = float(direct[0])
    draw = float(direct[1])
    home_win = float(direct[2])

    if neutral:
        # Evaluate the reverse ordering too. This removes arbitrary first-listed
        # "home" effects from neutral tournament matches.
        reverse = _ordered_probability_vector(
            away_team,
            home_team,
            tournament_weight,
            neutral,
        )

        home_win = (home_win + float(reverse[0])) / 2.0
        draw = (draw + float(reverse[1])) / 2.0
        away_win = (away_win + float(reverse[2])) / 2.0

    total = home_win + draw + away_win
    if total <= 0:
        raise ValueError('Model returned an invalid probability vector.')

    return (
        home_win / total,
        draw / total,
        away_win / total,
    )


def get_match_probabilities(
    home_team,
    away_team,
    tournament_weight=1.0,
    neutral=True,
):
    home_win, draw, away_win = _cached_team_probabilities(
        home_team,
        away_team,
        float(tournament_weight),
        bool(neutral),
    )

    return {
        'home_team': home_team,
        'away_team': away_team,
        'home_win': home_win,
        'draw': draw,
        'away_win': away_win,
    }


# -------------------------------------------------------------------------
# Team-specific prediction validation.
# -------------------------------------------------------------------------

example_fixtures = [
    ('Spain', 'Brazil'),
    ('France', 'Argentina'),
    ('England', 'Germany'),
    ('Japan', 'South Korea'),
]

prediction_example_rows = []

for home_team, away_team in example_fixtures:
    if home_team not in available_teams or away_team not in available_teams:
        continue

    prediction = get_match_probabilities(
        home_team,
        away_team,
        tournament_weight=1.0,
        neutral=True,
    )

    prediction_example_rows.append({
        'home_team': home_team,
        'away_team': away_team,
        'home_elo': float(team_state_lookup.loc[home_team, 'current_elo']),
        'away_elo': float(team_state_lookup.loc[away_team, 'current_elo']),
        'home_form_goals_for': float(
            team_state_lookup.loc[home_team, 'current_form_goals_for']
        ),
        'away_form_goals_for': float(
            team_state_lookup.loc[away_team, 'current_form_goals_for']
        ),
        'home_win_probability': prediction['home_win'],
        'draw_probability': prediction['draw'],
        'away_win_probability': prediction['away_win'],
        'probability_sum': (
            prediction['home_win']
            + prediction['draw']
            + prediction['away_win']
        ),
    })

team_specific_prediction_examples = pd.DataFrame(
    prediction_example_rows
)
team_specific_prediction_examples.to_csv(
    outputs_dir / 'team_specific_prediction_examples.csv',
    index=False,
)

if not np.allclose(
    team_specific_prediction_examples['probability_sum'],
    1.0,
    atol=1e-8,
):
    raise ValueError('Team-specific probabilities do not sum to one.')

# Neutral-order symmetry check. Swapping the team names should swap their win
# probabilities while leaving the draw probability unchanged.
symmetry_rows = []
for home_team, away_team in example_fixtures:
    if home_team not in available_teams or away_team not in available_teams:
        continue

    forward = get_match_probabilities(home_team, away_team, 1.0, True)
    reverse = get_match_probabilities(away_team, home_team, 1.0, True)

    symmetry_rows.append({
        'fixture': f'{home_team} vs {away_team}',
        'home_swap_difference': abs(
            forward['home_win'] - reverse['away_win']
        ),
        'away_swap_difference': abs(
            forward['away_win'] - reverse['home_win']
        ),
        'draw_swap_difference': abs(
            forward['draw'] - reverse['draw']
        ),
    })

team_prediction_validation = pd.DataFrame(symmetry_rows)
team_prediction_validation.to_csv(
    outputs_dir / 'team_prediction_validation.csv',
    index=False,
)

if (
    team_prediction_validation[[
        'home_swap_difference',
        'away_swap_difference',
        'draw_swap_difference',
    ]].to_numpy().max() > 1e-10
):
    raise ValueError('Neutral match predictions are not order-symmetric.')

print('\nTeam-specific prediction examples')
display(team_specific_prediction_examples)


# -------------------------------------------------------------------------
# Configurable 48-team-style tournament simulation.
# -------------------------------------------------------------------------

def build_seeded_groups(
    teams,
    group_count=12,
    group_size=4,
):
    teams = list(teams)
    expected_teams = group_count * group_size

    if len(teams) != expected_teams:
        raise ValueError(
            f'Expected {expected_teams} teams for '
            f'{group_count} groups of {group_size}; received {len(teams)}.'
        )
    if len(set(teams)) != len(teams):
        raise ValueError('Tournament team list contains duplicates.')

    for team in teams:
        _validate_team_name(team)

    # Rank teams by the current Elo snapshot, then distribute one team from
    # each pot to every group. Alternating pot direction produces a simple
    # snake-seeded demonstration draw.
    ranked = sorted(
        teams,
        key=lambda team: float(team_state_lookup.loc[team, 'current_elo']),
        reverse=True,
    )

    pots = [
        ranked[index * group_count:(index + 1) * group_count]
        for index in range(group_size)
    ]

    groups = {
        f'Group {chr(65 + index)}': []
        for index in range(group_count)
    }
    group_names = list(groups)

    for pot_number, pot in enumerate(pots):
        order = group_names if pot_number % 2 == 0 else list(reversed(group_names))
        for group_name, team in zip(order, pot):
            groups[group_name].append(team)

    return groups


def _sample_regulation_outcome(home_team, away_team, rng):
    probabilities = get_match_probabilities(
        home_team,
        away_team,
        tournament_weight=1.0,
        neutral=True,
    )

    return rng.choice(
        ['home', 'draw', 'away'],
        p=[
            probabilities['home_win'],
            probabilities['draw'],
            probabilities['away_win'],
        ],
    )


def simulate_group(group_name, teams, rng):
    points = {team: 0 for team in teams}

    for home_team, away_team in combinations(teams, 2):
        outcome = _sample_regulation_outcome(home_team, away_team, rng)

        if outcome == 'home':
            points[home_team] += 3
        elif outcome == 'away':
            points[away_team] += 3
        else:
            points[home_team] += 1
            points[away_team] += 1

    standings = pd.DataFrame([
        {
            'group': group_name,
            'team': team,
            'points': points[team],
            'elo_tiebreak': float(
                team_state_lookup.loc[team, 'current_elo']
            ),
        }
        for team in teams
    ])

    # The model predicts outcome class, not scoreline. Elo is therefore used
    # as a documented simplified tiebreak after group points.
    standings = (
        standings
        .sort_values(
            ['points', 'elo_tiebreak'],
            ascending=[False, False],
        )
        .reset_index(drop=True)
    )
    standings['group_position'] = np.arange(1, len(standings) + 1)

    return standings


def simulate_knockout_match(team_a, team_b, rng):
    probabilities = get_match_probabilities(
        team_a,
        team_b,
        tournament_weight=1.0,
        neutral=True,
    )

    regulation = rng.choice(
        ['team_a', 'draw', 'team_b'],
        p=[
            probabilities['home_win'],
            probabilities['draw'],
            probabilities['away_win'],
        ],
    )

    if regulation == 'team_a':
        return team_a
    if regulation == 'team_b':
        return team_b

    # If regulation is drawn, reuse the model's relative non-draw strength
    # as a simple extra-time/penalty advancement probability.
    non_draw_total = (
        probabilities['home_win']
        + probabilities['away_win']
    )

    if non_draw_total <= 0:
        team_a_advance = 0.5
    else:
        team_a_advance = (
            probabilities['home_win'] / non_draw_total
        )

    return rng.choice(
        [team_a, team_b],
        p=[team_a_advance, 1.0 - team_a_advance],
    )


def build_round_of_32_pairs(qualifiers):
    remaining = (
        qualifiers
        .sort_values(
            ['group_position', 'points', 'elo_tiebreak'],
            ascending=[True, False, False],
        )
        .to_dict('records')
    )

    pairs = []

    while remaining:
        high_seed = remaining.pop(0)

        opponent_index = None
        for candidate_index in range(len(remaining) - 1, -1, -1):
            if (
                remaining[candidate_index]['group']
                != high_seed['group']
            ):
                opponent_index = candidate_index
                break

        if opponent_index is None:
            opponent_index = len(remaining) - 1

        low_seed = remaining.pop(opponent_index)
        pairs.append((high_seed['team'], low_seed['team']))

    return pairs


def run_single_tournament(groups, rng):
    group_tables = []

    for group_name, teams in groups.items():
        group_tables.append(
            simulate_group(group_name, teams, rng)
        )

    group_table = pd.concat(group_tables, ignore_index=True)

    automatic_qualifiers = group_table.loc[
        group_table['group_position'].le(2)
    ].copy()

    best_thirds = (
        group_table.loc[group_table['group_position'].eq(3)]
        .sort_values(
            ['points', 'elo_tiebreak'],
            ascending=[False, False],
        )
        .head(8)
        .copy()
    )

    qualifiers = pd.concat(
        [automatic_qualifiers, best_thirds],
        ignore_index=True,
    )

    if len(qualifiers) != 32:
        raise ValueError(
            f'Expected 32 knockout qualifiers, found {len(qualifiers)}.'
        )

    stage_reached = {
        team: 'Group'
        for teams in groups.values()
        for team in teams
    }

    for team in qualifiers['team']:
        stage_reached[team] = 'R32'

    pairs = build_round_of_32_pairs(qualifiers)

    round_names = ['R32', 'R16', 'QF', 'SF', 'Final']
    next_stage = {
        'R32': 'R16',
        'R16': 'QF',
        'QF': 'SF',
        'SF': 'Final',
        'Final': 'Champion',
    }

    current_pairs = pairs

    for round_name in round_names:
        winners = []

        for team_a, team_b in current_pairs:
            winner = simulate_knockout_match(
                team_a,
                team_b,
                rng,
            )
            winners.append(winner)
            stage_reached[winner] = next_stage[round_name]

        if len(winners) == 1:
            champion = winners[0]
            break

        current_pairs = [
            (winners[index], winners[index + 1])
            for index in range(0, len(winners), 2)
        ]

    return champion, stage_reached


def run_monte_carlo_tournament(
    groups,
    iterations=2000,
    seed=42,
):
    if iterations <= 0:
        raise ValueError('iterations must be positive.')

    all_teams = [
        team
        for teams in groups.values()
        for team in teams
    ]

    if len(all_teams) != 48 or len(set(all_teams)) != 48:
        raise ValueError(
            'This tournament configuration expects 48 unique teams.'
        )

    rng = np.random.default_rng(seed)
    stage_counts = {
        team: Counter()
        for team in all_teams
    }

    for _ in range(iterations):
        champion, stage_reached = run_single_tournament(
            groups,
            rng,
        )

        for team, stage in stage_reached.items():
            stage_counts[team][stage] += 1

    ordered_stages = ['R32', 'R16', 'QF', 'SF', 'Final', 'Champion']

    rows = []
    for team in all_teams:
        counts = stage_counts[team]

        # Reaching a later stage implies reaching all preceding knockout stages.
        champion_count = counts['Champion']
        final_count = counts['Final'] + champion_count
        sf_count = counts['SF'] + final_count
        qf_count = counts['QF'] + sf_count
        r16_count = counts['R16'] + qf_count
        r32_count = counts['R32'] + r16_count

        cumulative = {
            'R32': r32_count,
            'R16': r16_count,
            'QF': qf_count,
            'SF': sf_count,
            'Final': final_count,
            'Champion': champion_count,
        }

        rows.append({
            'team': team,
            'current_elo': float(
                team_state_lookup.loc[team, 'current_elo']
            ),
            **{
                f'{stage.lower()}_probability':
                    cumulative[stage] / iterations
                for stage in ordered_stages
            },
        })

    results = (
        pd.DataFrame(rows)
        .sort_values(
            ['champion_probability', 'final_probability'],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return results


# Default demonstration selection:
# top 48 teams in the current Elo state snapshot. This is deliberately NOT
# labelled as the official World Cup participant list. Users can supply any
# 48 teams available in team_state_snapshot.
default_tournament_teams = (
    team_state_snapshot
    .head(48)['team']
    .tolist()
)

default_groups = build_seeded_groups(
    default_tournament_teams,
    group_count=12,
    group_size=4,
)

group_configuration_rows = []
for group_name, teams in default_groups.items():
    for seed_position, team in enumerate(teams, start=1):
        group_configuration_rows.append({
            'group': group_name,
            'seed_position': seed_position,
            'team': team,
            'current_elo': float(
                team_state_lookup.loc[team, 'current_elo']
            ),
            'selection_note': 'top_48_by_current_elo_demo',
        })

tournament_group_configuration = pd.DataFrame(
    group_configuration_rows
)
tournament_group_configuration.to_csv(
    outputs_dir / 'tournament_group_configuration.csv',
    index=False,
)

simulation_iterations = 2000
simulation_seed = 42

tournament_simulation_results = run_monte_carlo_tournament(
    default_groups,
    iterations=simulation_iterations,
    seed=simulation_seed,
)
tournament_simulation_results.to_csv(
    outputs_dir / 'tournament_simulation_results.csv',
    index=False,
)

champion_probability_sum = float(
    tournament_simulation_results['champion_probability'].sum()
)

simulation_validation = pd.DataFrame([
    ('configured_teams', len(default_tournament_teams)),
    ('configured_groups', len(default_groups)),
    ('teams_per_group', 4),
    ('automatic_top_two_qualifiers', 24),
    ('best_third_place_qualifiers', 8),
    ('round_of_32_teams', 32),
    ('simulation_iterations', simulation_iterations),
    ('simulation_seed', simulation_seed),
    ('champion_probability_sum', champion_probability_sum),
    ('cached_probability_matchups', _cached_team_probabilities.cache_info().currsize),
], columns=['metric', 'value'])

simulation_validation.to_csv(
    outputs_dir / 'tournament_simulation_validation.csv',
    index=False,
)

if not np.isclose(champion_probability_sum, 1.0, atol=1e-9):
    raise ValueError(
        'Champion probabilities do not sum to one.'
    )

figures_dir = outputs_dir / 'figures'
figures_dir.mkdir(exist_ok=True)

top_champions = tournament_simulation_results.head(15)

plt.figure(figsize=(9, 6))
plt.barh(
    top_champions['team'][::-1],
    top_champions['champion_probability'][::-1] * 100,
)
plt.xlabel('Champion probability (%)')
plt.ylabel('Team')
plt.title(
    f'Top simulated champions ({simulation_iterations:,} iterations)'
)
plt.tight_layout()
plt.savefig(
    figures_dir / 'champion_probabilities.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

print('\nSelected probability model:', selected_probability_model_name)
print('Current team states:', len(team_state_snapshot))
print('Default demonstration teams:', len(default_tournament_teams))
print('Monte Carlo iterations:', simulation_iterations)
print('\nTop simulated champions')
display(tournament_simulation_results.head(15))


# -------------------------------------------------------------------------
# Final release validation summary
# -------------------------------------------------------------------------

selected_row = probability_model_comparison.iloc[0]
draw_row = draw_probability_diagnostic.iloc[0]

final_run_validation = pd.DataFrame([
    ('validated_completed_matches', len(df_matches)),
    ('modern_model_matches', len(df_clean)),
    ('training_matches', training_rows),
    ('test_matches', test_rows),
    ('test_draws', int((np.asarray(y_test_probability) == 1).sum())),
    ('test_draw_rate', float((np.asarray(y_test_probability) == 1).mean())),
    ('selected_probability_model', selected_probability_model_name),
    ('selected_log_loss', float(selected_row['log_loss'])),
    ('selected_multiclass_brier', float(selected_row['multiclass_brier'])),
    ('selected_argmax_draw_recall', float(draw_row['argmax_draw_recall'])),
    ('selected_mean_draw_probability', float(draw_row['mean_predicted_draw_probability'])),
    ('team_state_count', len(team_state_snapshot)),
    ('tournament_demo_teams', len(default_tournament_teams)),
    ('simulation_iterations', simulation_iterations),
    ('simulation_seed', simulation_seed),
    ('champion_probability_sum', champion_probability_sum),
], columns=['metric', 'value'])

final_run_validation.to_csv(
    outputs_dir / 'final_run_validation.csv',
    index=False,
)

print('\nFinal release checks')
display(final_run_validation)


## Final interpretation and limitations

The selected model is chosen for probability quality rather than hard-class draw recall. A draw can receive meaningful probability without being the single most likely outcome, which matters because the tournament simulator samples from the full probability vector.

The tournament component is a modelling approximation. It uses a configurable 48-team field, points followed by Elo for group ties, and a seeded knockout pairing rather than reproducing every official FIFA tiebreak and bracket rule. The bundled demonstration uses the top 48 teams in the current Elo snapshot, not an authoritative participant list.

The implemented feature set is intentionally narrower than some approaches discussed in the original literature review: it uses historical results, recent form, Elo, tournament importance and neutral venue status. Bookmaker odds, FIFA rankings, player market values, geography, climate, travel and GDP are not model inputs in the final pipeline.


## Commit 12: Retrospective 2026 World Cup validation

This section runs only after the existing Commit 11 modelling pipeline. The selected probability model is frozen before World Cup outcomes are loaded.

The classifier is trained on matches before 2023. Model selection and the frozen Elo/form snapshot use completed project data through 31 March 2026. The World Cup began on 11 June 2026, leaving a 72-day feature-freshness gap.

All 104 tournament fixtures are evaluated from the same frozen pre-tournament state. This is a retrospective backtest rather than a sequential in-tournament updating experiment. Penalty-shootout ties remain draws for three-way scoring, with advancement stored separately.


In [ ]:
# -------------------------------------------------------------------------
# Commit 12: frozen-model retrospective backtest on the completed 2026 World Cup
# -------------------------------------------------------------------------

from sklearn.metrics import accuracy_score, log_loss, confusion_matrix

world_cup_actual = pd.read_csv(
    project_root / 'data' / 'world_cup_2026_actual_results.csv',
    parse_dates=['date'],
)
world_cup_groups = pd.read_csv(
    project_root / 'data' / 'world_cup_2026_groups.csv'
)

expected_selected_model = 'Calibrated Linear SVM'
if selected_probability_model_name != expected_selected_model:
    raise ValueError(
        'Commit 12 must keep the Commit 11 model frozen. '
        f'Expected {expected_selected_model}, found {selected_probability_model_name}.'
    )

model_training_end = df_model.loc[
    df_model['date'] < evaluation_cutoff,
    'date',
].max()
model_holdout_end = df_model.loc[
    df_model['date'] >= evaluation_cutoff,
    'date',
].max()
state_data_end = df_clean['date'].max()
world_cup_start = world_cup_actual['date'].min()
world_cup_end = world_cup_actual['date'].max()

if state_data_end >= world_cup_start:
    raise ValueError('World Cup result leakage detected in the frozen state.')
if len(world_cup_actual) != 104:
    raise ValueError(f'Expected 104 World Cup matches, found {len(world_cup_actual)}.')
if world_cup_actual['match_id'].nunique() != 104:
    raise ValueError('World Cup match IDs are not unique.')

world_cup_teams = sorted(
    set(world_cup_actual['team_a'])
    | set(world_cup_actual['team_b'])
)
if len(world_cup_teams) != 48:
    raise ValueError(f'Expected 48 World Cup teams, found {len(world_cup_teams)}.')

unresolved_world_cup_teams = sorted(
    set(world_cup_teams) - available_teams
)
if unresolved_world_cup_teams:
    raise ValueError(
        'World Cup team names missing from the frozen state snapshot: '
        + ', '.join(unresolved_world_cup_teams)
    )

world_cup_matches_in_model_data = int(
    (
        df_model['date'].between(
            world_cup_start,
            world_cup_end,
            inclusive='both',
        )
        & df_model['tournament'].astype(str).str.contains(
            'FIFA World Cup',
            case=False,
            na=False,
        )
    ).sum()
)
if world_cup_matches_in_model_data != 0:
    raise ValueError('Completed 2026 World Cup matches entered model-development data.')


def frozen_fixture_probabilities(team_a, team_b, neutral):
    prediction = get_match_probabilities(
        team_a,
        team_b,
        tournament_weight=1.0,
        neutral=bool(neutral),
    )
    return (
        float(prediction['home_win']),
        float(prediction['draw']),
        float(prediction['away_win']),
    )


outcome_to_label = {
    'team_b_win': 0,
    'draw': 1,
    'team_a_win': 2,
}
label_to_outcome = {
    0: 'team_b_win',
    1: 'draw',
    2: 'team_a_win',
}

backtest_rows = []

for match in world_cup_actual.itertuples(index=False):
    p_team_a_win, p_draw, p_team_b_win = frozen_fixture_probabilities(
        match.team_a,
        match.team_b,
        match.neutral,
    )

    probabilities = np.array([
        p_team_b_win,
        p_draw,
        p_team_a_win,
    ])

    if not np.isclose(probabilities.sum(), 1.0, atol=1e-8):
        raise ValueError(f'Probability sum failed for {match.match_id}.')

    predicted_label = int(
        class_labels[np.argmax(probabilities)]
    )
    actual_label = outcome_to_label[match.outcome]

    backtest_rows.append({
        'match_id': match.match_id,
        'match_number': int(match.match_number),
        'date': match.date,
        'stage': match.stage,
        'group': match.group,
        'team_a': match.team_a,
        'team_b': match.team_b,
        'neutral': bool(match.neutral),
        'final_score_a': int(match.final_score_a),
        'final_score_b': int(match.final_score_b),
        'decided_by': match.decided_by,
        'advanced_team': match.advanced_team,
        'actual_outcome': match.outcome,
        'actual_label': actual_label,
        'p_team_a_win': p_team_a_win,
        'p_draw': p_draw,
        'p_team_b_win': p_team_b_win,
        'predicted_outcome': label_to_outcome[predicted_label],
        'predicted_label': predicted_label,
        'correct_argmax': int(predicted_label == actual_label),
    })

world_cup_match_predictions = pd.DataFrame(backtest_rows)
if len(world_cup_match_predictions) != 104:
    raise ValueError('Backtest prediction row count changed unexpectedly.')

world_cup_match_predictions.to_csv(
    outputs_dir / 'world_cup_2026_match_predictions.csv',
    index=False,
)

actual_labels_wc = world_cup_match_predictions['actual_label'].to_numpy(dtype=int)
predicted_labels_wc = world_cup_match_predictions['predicted_label'].to_numpy(dtype=int)
probabilities_wc = world_cup_match_predictions[
    ['p_team_b_win', 'p_draw', 'p_team_a_win']
].to_numpy(dtype=float)


def score_probability_frame(frame):
    y_true = frame['actual_label'].to_numpy(dtype=int)
    y_pred = frame['predicted_label'].to_numpy(dtype=int)
    probs = frame[
        ['p_team_b_win', 'p_draw', 'p_team_a_win']
    ].to_numpy(dtype=float)

    recalls = recall_score(
        y_true,
        y_pred,
        labels=class_labels,
        average=None,
        zero_division=0,
    )

    return {
        'matches': len(frame),
        'accuracy': accuracy_score(y_true, y_pred),
        'log_loss': log_loss(y_true, probs, labels=class_labels),
        'multiclass_brier': multiclass_brier_score(
            y_true,
            probs,
            class_labels,
        ),
        'actual_draw_rate': float((y_true == 1).mean()),
        'mean_predicted_draw_probability': float(probs[:, 1].mean()),
        'argmax_draw_prediction_rate': float((y_pred == 1).mean()),
        'argmax_draw_recall': float(recalls[1]),
    }


metric_frames = {
    'Overall': world_cup_match_predictions,
    'Group stage': world_cup_match_predictions.loc[
        world_cup_match_predictions['stage'].eq('group')
    ],
    'Knockout stage': world_cup_match_predictions.loc[
        ~world_cup_match_predictions['stage'].eq('group')
    ],
}

world_cup_match_metrics = pd.DataFrame([
    {'split': split_name, **score_probability_frame(frame)}
    for split_name, frame in metric_frames.items()
])
world_cup_match_metrics.to_csv(
    outputs_dir / 'world_cup_2026_match_metrics.csv',
    index=False,
)

# Historical class-prior baseline, included only as a reference benchmark.
baseline_raw = dummy_model.predict_proba(
    np.zeros(
        (len(world_cup_match_predictions), len(features_upgraded))
    )
)
baseline_probs = align_probability_columns(
    dummy_model,
    baseline_raw,
    class_labels,
)
baseline_pred = class_labels[
    np.argmax(baseline_probs, axis=1)
]

model_vs_baseline = pd.DataFrame([
    {
        'model': selected_probability_model_name,
        'accuracy': accuracy_score(actual_labels_wc, predicted_labels_wc),
        'log_loss': log_loss(
            actual_labels_wc,
            probabilities_wc,
            labels=class_labels,
        ),
        'multiclass_brier': multiclass_brier_score(
            actual_labels_wc,
            probabilities_wc,
            class_labels,
        ),
    },
    {
        'model': 'Historical class-prior baseline',
        'accuracy': accuracy_score(actual_labels_wc, baseline_pred),
        'log_loss': log_loss(
            actual_labels_wc,
            baseline_probs,
            labels=class_labels,
        ),
        'multiclass_brier': multiclass_brier_score(
            actual_labels_wc,
            baseline_probs,
            class_labels,
        ),
    },
])
model_vs_baseline.to_csv(
    outputs_dir / 'world_cup_2026_model_vs_baseline.csv',
    index=False,
)

# Secondary confusion-matrix diagnostic.
wc_confusion = confusion_matrix(
    actual_labels_wc,
    predicted_labels_wc,
    labels=class_labels,
)
wc_confusion_rows = []
for actual_index, actual_label in enumerate(class_labels):
    for predicted_index, predicted_label in enumerate(class_labels):
        wc_confusion_rows.append({
            'actual_label': int(actual_label),
            'actual_name': class_names[int(actual_label)],
            'predicted_label': int(predicted_label),
            'predicted_name': class_names[int(predicted_label)],
            'count': int(
                wc_confusion[
                    actual_index,
                    predicted_index,
                ]
            ),
        })

pd.DataFrame(wc_confusion_rows).to_csv(
    outputs_dir / 'world_cup_2026_confusion_matrix.csv',
    index=False,
)

# -------------------------------------------------------------------------
# 5,000-resample bootstrap intervals for the 104-match test
# -------------------------------------------------------------------------

bootstrap_iterations = 5000
bootstrap_seed = 2026
bootstrap_rng = np.random.default_rng(bootstrap_seed)

bootstrap_accuracy = np.empty(bootstrap_iterations)
bootstrap_log_loss = np.empty(bootstrap_iterations)
bootstrap_brier = np.empty(bootstrap_iterations)

for bootstrap_index in range(bootstrap_iterations):
    sample_index = bootstrap_rng.integers(
        0,
        len(actual_labels_wc),
        size=len(actual_labels_wc),
    )

    sample_y = actual_labels_wc[sample_index]
    sample_pred = predicted_labels_wc[sample_index]
    sample_probs = probabilities_wc[sample_index]

    bootstrap_accuracy[bootstrap_index] = accuracy_score(
        sample_y,
        sample_pred,
    )
    bootstrap_log_loss[bootstrap_index] = log_loss(
        sample_y,
        sample_probs,
        labels=class_labels,
    )
    bootstrap_brier[bootstrap_index] = multiclass_brier_score(
        sample_y,
        sample_probs,
        class_labels,
    )

overall_metrics = world_cup_match_metrics.loc[
    world_cup_match_metrics['split'].eq('Overall')
].iloc[0]

bootstrap_metric_map = {
    'accuracy': (
        float(overall_metrics['accuracy']),
        bootstrap_accuracy,
    ),
    'log_loss': (
        float(overall_metrics['log_loss']),
        bootstrap_log_loss,
    ),
    'multiclass_brier': (
        float(overall_metrics['multiclass_brier']),
        bootstrap_brier,
    ),
}

bootstrap_rows = []
for metric_name, (estimate, values) in bootstrap_metric_map.items():
    lower, upper = np.quantile(values, [0.025, 0.975])
    bootstrap_rows.append({
        'metric': metric_name,
        'estimate': estimate,
        'ci_lower_95': float(lower),
        'ci_upper_95': float(upper),
        'bootstrap_iterations': bootstrap_iterations,
        'bootstrap_seed': bootstrap_seed,
    })

world_cup_bootstrap_metrics = pd.DataFrame(bootstrap_rows)
world_cup_bootstrap_metrics.to_csv(
    outputs_dir / 'world_cup_2026_bootstrap_metrics.csv',
    index=False,
)

# -------------------------------------------------------------------------
# Actual 48-team field tournament simulation
# -------------------------------------------------------------------------

actual_groups = {
    f'Group {group_name}': (
        group_frame
        .sort_values('listed_position')['team']
        .tolist()
    )
    for group_name, group_frame in world_cup_groups.groupby(
        'group',
        sort=True,
    )
}

if len(actual_groups) != 12:
    raise ValueError(f'Expected 12 actual groups, found {len(actual_groups)}.')
if sum(len(teams) for teams in actual_groups.values()) != 48:
    raise ValueError('Actual group table does not contain exactly 48 team slots.')

post_tournament_simulation_iterations = 2000
post_tournament_simulation_seed = 2026

# Precompute every neutral pair in one batched model call. The Commit 10
# simulator calls predict_proba one fixture at a time, which is convenient
# interactively but unnecessarily slow for a second 2,000-run validation.
from itertools import combinations

actual_field_teams = sorted(
    team for teams in actual_groups.values() for team in teams
)
pair_list = list(combinations(actual_field_teams, 2))

forward_rows = []
reverse_rows = []
for team_a, team_b in pair_list:
    forward_rows.append(
        build_match_features(
            team_a,
            team_b,
            tournament_weight=1.0,
            neutral=True,
        ).iloc[0]
    )
    reverse_rows.append(
        build_match_features(
            team_b,
            team_a,
            tournament_weight=1.0,
            neutral=True,
        ).iloc[0]
    )

forward_features = pd.DataFrame(forward_rows, columns=features_upgraded)
reverse_features = pd.DataFrame(reverse_rows, columns=features_upgraded)

forward_probs = align_probability_columns(
    selected_probability_model,
    selected_probability_model.predict_proba(forward_features),
    class_labels,
)
reverse_probs = align_probability_columns(
    selected_probability_model,
    selected_probability_model.predict_proba(reverse_features),
    class_labels,
)

fast_probability_lookup = {}
for index, (team_a, team_b) in enumerate(pair_list):
    # forward classes: away win, draw, home win
    p_a_win = (
        float(forward_probs[index, 2])
        + float(reverse_probs[index, 0])
    ) / 2.0
    p_draw = (
        float(forward_probs[index, 1])
        + float(reverse_probs[index, 1])
    ) / 2.0
    p_b_win = (
        float(forward_probs[index, 0])
        + float(reverse_probs[index, 2])
    ) / 2.0
    total = p_a_win + p_draw + p_b_win
    fast_probability_lookup[(team_a, team_b)] = (
        p_a_win / total,
        p_draw / total,
        p_b_win / total,
    )


def fast_pair_probabilities(team_a, team_b):
    if team_a < team_b:
        return fast_probability_lookup[(team_a, team_b)]
    p_b_win, p_draw, p_a_win = fast_probability_lookup[(team_b, team_a)]
    return p_a_win, p_draw, p_b_win


actual_field_elo = {
    team: float(team_state_lookup.loc[team, 'current_elo'])
    for team in actual_field_teams
}
actual_group_pairs = {
    group_name: list(combinations(teams, 2))
    for group_name, teams in actual_groups.items()
}


def fast_group_standings(group_name, teams, rng):
    points = {team: 0 for team in teams}

    for team_a, team_b in actual_group_pairs[group_name]:
        p_a, p_draw, _ = fast_pair_probabilities(team_a, team_b)
        draw_value = rng.random()
        if draw_value < p_a:
            points[team_a] += 3
        elif draw_value < p_a + p_draw:
            points[team_a] += 1
            points[team_b] += 1
        else:
            points[team_b] += 3

    ordered = sorted(
        teams,
        key=lambda team: (
            points[team],
            actual_field_elo[team],
        ),
        reverse=True,
    )

    return [
        {
            'group': group_name,
            'team': team,
            'points': points[team],
            'elo_tiebreak': actual_field_elo[team],
            'group_position': position,
        }
        for position, team in enumerate(ordered, start=1)
    ]


def fast_knockout_winner(team_a, team_b, rng):
    p_a, p_draw, p_b = fast_pair_probabilities(team_a, team_b)
    value = rng.random()

    if value < p_a:
        return team_a
    if value >= p_a + p_draw:
        return team_b

    non_draw = p_a + p_b
    if non_draw <= 0:
        return team_a if rng.random() < 0.5 else team_b
    return team_a if rng.random() < (p_a / non_draw) else team_b


def fast_round_of_32_pairs(qualifiers):
    remaining = sorted(
        qualifiers,
        key=lambda row: (
            row['group_position'],
            -row['points'],
            -row['elo_tiebreak'],
        ),
    )
    pairs = []

    while remaining:
        high_seed = remaining.pop(0)
        opponent_index = None
        for candidate_index in range(len(remaining) - 1, -1, -1):
            if remaining[candidate_index]['group'] != high_seed['group']:
                opponent_index = candidate_index
                break
        if opponent_index is None:
            opponent_index = len(remaining) - 1
        low_seed = remaining.pop(opponent_index)
        pairs.append((high_seed['team'], low_seed['team']))

    return pairs


def run_fast_actual_field_simulation(iterations=2000, seed=2026):
    rng = np.random.default_rng(seed)
    reached = {
        team: {
            'R32': 0,
            'R16': 0,
            'QF': 0,
            'SF': 0,
            'Final': 0,
            'Champion': 0,
        }
        for team in actual_field_teams
    }

    for _ in range(iterations):
        all_standings = []
        for group_name, teams in actual_groups.items():
            all_standings.extend(
                fast_group_standings(group_name, teams, rng)
            )

        automatic = [
            row for row in all_standings
            if row['group_position'] <= 2
        ]
        thirds = sorted(
            [
                row for row in all_standings
                if row['group_position'] == 3
            ],
            key=lambda row: (
                row['points'],
                row['elo_tiebreak'],
            ),
            reverse=True,
        )[:8]
        qualifiers = automatic + thirds

        for row in qualifiers:
            reached[row['team']]['R32'] += 1

        pairs = fast_round_of_32_pairs(qualifiers)
        current_teams = []
        for team_a, team_b in pairs:
            winner = fast_knockout_winner(team_a, team_b, rng)
            current_teams.append(winner)
            reached[winner]['R16'] += 1

        stage_sequence = [
            ('QF', 16),
            ('SF', 8),
            ('Final', 4),
            ('Champion', 2),
        ]

        for stage_name, expected_input in stage_sequence:
            if len(current_teams) != expected_input:
                raise ValueError(
                    f'Unexpected knockout field size before {stage_name}: '
                    f'{len(current_teams)}.'
                )
            winners = []
            for pair_index in range(0, len(current_teams), 2):
                winner = fast_knockout_winner(
                    current_teams[pair_index],
                    current_teams[pair_index + 1],
                    rng,
                )
                winners.append(winner)
                reached[winner][stage_name] += 1
            current_teams = winners

    rows = []
    for team in actual_field_teams:
        rows.append({
            'team': team,
            'current_elo': actual_field_elo[team],
            'r32_probability': reached[team]['R32'] / iterations,
            'r16_probability': reached[team]['R16'] / iterations,
            'qf_probability': reached[team]['QF'] / iterations,
            'sf_probability': reached[team]['SF'] / iterations,
            'final_probability': reached[team]['Final'] / iterations,
            'champion_probability': reached[team]['Champion'] / iterations,
        })

    results = pd.DataFrame(rows).sort_values(
        ['champion_probability', 'final_probability'],
        ascending=False,
    ).reset_index(drop=True)

    if not np.isclose(results['champion_probability'].sum(), 1.0, atol=1e-9):
        raise ValueError('Actual-field champion probabilities do not sum to one.')

    return results


world_cup_pre_tournament_simulation = run_fast_actual_field_simulation(
    iterations=post_tournament_simulation_iterations,
    seed=post_tournament_simulation_seed,
)

stage_order = ['Group', 'R32', 'R16', 'QF', 'SF', 'Final', 'Champion']
stage_numeric = {
    stage: index
    for index, stage in enumerate(stage_order)
}

actual_stage = {
    team: 'Group'
    for team in world_cup_teams
}
stage_to_label = {
    'round_of_32': 'R32',
    'round_of_16': 'R16',
    'quarter_final': 'QF',
    'semi_final': 'SF',
    'final': 'Final',
}

for stage_name, stage_label in stage_to_label.items():
    stage_matches = world_cup_actual.loc[
        world_cup_actual['stage'].eq(stage_name)
    ]
    participants = set(stage_matches['team_a']) | set(stage_matches['team_b'])

    for team in participants:
        if stage_numeric[stage_label] > stage_numeric[actual_stage[team]]:
            actual_stage[team] = stage_label

final_match = world_cup_actual.loc[
    world_cup_actual['stage'].eq('final')
].iloc[0]
actual_champion = final_match['advanced_team']
actual_runner_up = (
    final_match['team_b']
    if actual_champion == final_match['team_a']
    else final_match['team_a']
)
actual_stage[actual_champion] = 'Champion'
actual_stage[actual_runner_up] = 'Final'

third_place_match = world_cup_actual.loc[
    world_cup_actual['stage'].eq('third_place')
].iloc[0]
actual_third = third_place_match['advanced_team']
actual_fourth = (
    third_place_match['team_b']
    if actual_third == third_place_match['team_a']
    else third_place_match['team_a']
)


def actual_finish_label(team):
    if team == actual_champion:
        return 'Champion'
    if team == actual_runner_up:
        return 'Runner-up'
    if team == actual_third:
        return 'Third'
    if team == actual_fourth:
        return 'Fourth'
    return actual_stage[team]


world_cup_expected_vs_actual = world_cup_pre_tournament_simulation.copy()
world_cup_expected_vs_actual['actual_stage_reached'] = (
    world_cup_expected_vs_actual['team'].map(actual_stage)
)
world_cup_expected_vs_actual['actual_finish'] = (
    world_cup_expected_vs_actual['team'].map(actual_finish_label)
)
world_cup_expected_vs_actual['actual_stage_numeric'] = (
    world_cup_expected_vs_actual[
        'actual_stage_reached'
    ].map(stage_numeric)
)
world_cup_expected_vs_actual['champion_probability_rank'] = (
    world_cup_expected_vs_actual['champion_probability']
    .rank(method='min', ascending=False)
    .astype(int)
)
world_cup_expected_vs_actual.to_csv(
    outputs_dir / 'world_cup_2026_expected_vs_actual.csv',
    index=False,
)

stage_probability_columns = {
    'R32': 'r32_probability',
    'R16': 'r16_probability',
    'QF': 'qf_probability',
    'SF': 'sf_probability',
    'Final': 'final_probability',
    'Champion': 'champion_probability',
}

stage_validation_rows = []
for stage_name, probability_column in stage_probability_columns.items():
    actual_binary = (
        world_cup_expected_vs_actual['actual_stage_numeric']
        >= stage_numeric[stage_name]
    ).astype(int)

    predicted_probability = world_cup_expected_vs_actual[
        probability_column
    ].astype(float)

    # The number of available places at each stage is fixed by the
    # tournament structure. Therefore the average predicted probability
    # across all 48 teams is mechanically tied to that slot count and is
    # not evidence of calibration. Validate the model at team level instead.
    slots = int(actual_binary.sum())

    predicted_top_k = set(
        world_cup_expected_vs_actual
        .assign(_stage_probability=predicted_probability)
        .nlargest(slots, '_stage_probability')['team']
    )
    actual_reached = set(
        world_cup_expected_vs_actual.loc[
            actual_binary.eq(1),
            'team',
        ]
    )

    top_k_capture_rate = (
        len(predicted_top_k & actual_reached) / slots
        if slots > 0
        else np.nan
    )

    stage_validation_rows.append({
        'stage': stage_name,
        'teams': len(actual_binary),
        'slots': slots,
        'stage_brier_score': float(
            np.mean(
                (predicted_probability - actual_binary) ** 2
            )
        ),
        'top_k_capture_rate': float(top_k_capture_rate),
    })

world_cup_stage_validation = pd.DataFrame(stage_validation_rows)
world_cup_stage_validation.to_csv(
    outputs_dir / 'world_cup_2026_stage_validation.csv',
    index=False,
)

# Small-sample draw calibration summary.
world_cup_draw_calibration = (
    world_cup_match_predictions
    .assign(
        draw_actual=lambda frame:
            frame['actual_label'].eq(1).astype(int),
        draw_bin=lambda frame:
            pd.qcut(
                frame['p_draw'],
                q=5,
                duplicates='drop',
            ).astype(str),
    )
    .groupby(
        'draw_bin',
        observed=True,
        as_index=False,
    )
    .agg(
        matches=('match_id', 'count'),
        mean_predicted_draw_probability=('p_draw', 'mean'),
        observed_draw_rate=('draw_actual', 'mean'),
    )
)
world_cup_draw_calibration.to_csv(
    outputs_dir / 'world_cup_2026_draw_calibration.csv',
    index=False,
)

freshness_gap_days = int((world_cup_start - state_data_end).days)

backtest_validation = pd.DataFrame([
    ('selected_model', selected_probability_model_name),
    ('model_training_end', model_training_end.strftime('%Y-%m-%d')),
    ('historical_holdout_end', model_holdout_end.strftime('%Y-%m-%d')),
    ('pre_tournament_state_data_end', state_data_end.strftime('%Y-%m-%d')),
    ('world_cup_start', world_cup_start.strftime('%Y-%m-%d')),
    ('world_cup_end', world_cup_end.strftime('%Y-%m-%d')),
    ('state_to_world_cup_gap_days', freshness_gap_days),
    ('world_cup_matches_evaluated', len(world_cup_actual)),
    ('world_cup_participants', len(world_cup_teams)),
    ('world_cup_matches_in_model_data', world_cup_matches_in_model_data),
    ('world_cup_outcomes_used_for_retraining', 0),
    ('world_cup_outcomes_used_for_model_reselection', 0),
    ('bootstrap_iterations', bootstrap_iterations),
    ('bootstrap_seed', bootstrap_seed),
    ('actual_field_simulation_iterations', post_tournament_simulation_iterations),
    ('actual_field_simulation_seed', post_tournament_simulation_seed),
], columns=['metric', 'value'])
backtest_validation.to_csv(
    outputs_dir / 'world_cup_2026_backtest_validation.csv',
    index=False,
)

# Figures
figures_dir = outputs_dir / 'figures'
figures_dir.mkdir(exist_ok=True)

plot_bootstrap = world_cup_bootstrap_metrics.copy()
x_positions = np.arange(len(plot_bootstrap))
estimates = plot_bootstrap['estimate'].to_numpy(dtype=float)
lower_error = estimates - plot_bootstrap['ci_lower_95'].to_numpy(dtype=float)
upper_error = plot_bootstrap['ci_upper_95'].to_numpy(dtype=float)

plt.figure(figsize=(8, 5))
plt.errorbar(
    x_positions,
    estimates,
    yerr=np.vstack([lower_error, upper_error]),
    fmt='o',
    capsize=5,
)
plt.xticks(x_positions, ['Accuracy', 'Log loss', 'Brier'])
plt.ylabel('Metric value')
plt.title('2026 World Cup backtest with 95% bootstrap intervals')
plt.tight_layout()
plt.savefig(
    figures_dir / 'world_cup_2026_probability_performance.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

plt.figure(figsize=(7, 5))
plt.plot(
    world_cup_draw_calibration['mean_predicted_draw_probability'],
    world_cup_draw_calibration['observed_draw_rate'],
    marker='o',
)
plt.plot([0, 0.5], [0, 0.5], linestyle='--')
plt.xlabel('Mean predicted draw probability')
plt.ylabel('Observed draw rate')
plt.title('2026 World Cup draw calibration (5 quantile bins)')
plt.tight_layout()
plt.savefig(
    figures_dir / 'world_cup_2026_draw_calibration.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

top_expected = world_cup_expected_vs_actual.head(16).copy()
top_expected['plot_label'] = (
    top_expected['team']
    + ' — '
    + top_expected['actual_finish']
)

plt.figure(figsize=(9, 7))
plt.barh(
    top_expected['plot_label'][::-1],
    top_expected['champion_probability'][::-1] * 100,
)
plt.xlabel('Frozen-model champion probability (%)')
plt.ylabel('Team and actual finish')
plt.title('Actual 2026 field: expected title probability vs actual finish')
plt.tight_layout()
plt.savefig(
    figures_dir / 'world_cup_2026_expected_vs_actual.png',
    dpi=150,
    bbox_inches='tight',
)
plt.close()

print('\n2026 World Cup retrospective match metrics')
display(world_cup_match_metrics)

print('\nBootstrap 95% intervals')
display(world_cup_bootstrap_metrics)

print('\nModel vs historical class-prior baseline')
display(model_vs_baseline)

print('\nActual-field tournament comparison')
display(
    world_cup_expected_vs_actual[[
        'team',
        'champion_probability',
        'champion_probability_rank',
        'actual_finish',
    ]].head(16)
)

print('\nLeakage/freeze audit')
display(backtest_validation)


## Post-tournament interpretation

This is a retrospective backtest of a frozen pre-tournament project state, not a claim that these predictions were timestamped and published before kickoff. The analysis is intentionally probability-first: log loss, Brier score and calibration are more relevant to the Monte Carlo use case than whether `Draw` is the single highest-probability class.

The 104-match sample is small, so the headline metrics are accompanied by 5,000-resample bootstrap intervals. The tournament-level comparison uses the actual 48-team group field but retains the project's approximate seeded knockout mapping and Elo-after-points group tiebreak.

Feature freshness remains a limitation. The frozen state ends on 31 March 2026, 72 days before the opening match, so later friendlies, qualifiers, final squad selections, injuries and tactical changes are deliberately excluded.
